In [2]:
import pandas as pd
import plotly.express as px

# Importing the data

In [3]:
# Demographic Snapshot (DOE)
doe = pd.read_csv("2019-20_Demographic_Snapshot_-_Borough_20251117.csv")

# New Capacity Program (SCA)
sca = pd.read_csv("New_Capacity_Program_By_Borough_20251117.csv")

# Check what the columns look like
print("DOE columns:", doe.columns.tolist())
print("SCA columns:", sca.columns.tolist())

doe.head()
sca.head()

DOE columns: ['Borough', 'Year', 'Total Enrollment', 'Grade 3K+PK (Half Day & Full Day)', 'Grade K', 'Grade 1', 'Grade 2', 'Grade 3', 'Grade 4', 'Grade 5', 'Grade 6', 'Grade 7', 'Grade 8', 'Grade 9', 'Grade 10', 'Grade 11', 'Grade 12', '# Female', '% Female', '# Male', '% Male', '# Asian', '% Asian', '# Black', '% Black', '# Hispanic', '% Hispanic', '# Multiple Race Categories Not Represented', '% Multiple Race Categories Not Represented', '# White', '% White', '# Students with Disabilities', '% Students with Disabilities', '# English Language Learners', '% English Language Learners', '# Poverty', '% Poverty', 'Economic Need Index']
SCA columns: ['DISTRICT', 'BOROUGH', 'SMALL PS # BLDGS', 'SMALL PS # SEATS', 'SMALL PS COST', 'PS/IS # BLDGS', 'PS/IS # SEATS', 'PS/IS COST', 'IS/HS # BLDGS', 'IS/HS # SEATS', 'IS/HS COST']


,DISTRICT,BOROUGH,SMALL PS # BLDGS,SMALL PS # SEATS,SMALL PS COST,PS/IS # BLDGS,PS/IS # SEATS,PS/IS COST,IS/HS # BLDGS,IS/HS # SEATS,IS/HS COST
0,1,MANHATTAN,0,0,0.00,0,0,0.00,0,0,0.0
1,2*,MANHATTAN,3,"1,368",71.53,2,"1,782",145.41,0,0,0.0
2,3,MANHATTAN,0,0,0.00,1,692,106.78,0,0,0.0
3,4,MANHATTAN,0,0,0.00,0,0,0.00,0,0,0.0
4,5,MANHATTAN,1,245,20.00,0,0,0.00,0,0,0.0


# Clean Demographics Snapshot Dataset

In [4]:
doe_small = doe[["Borough", "Year", "Total Enrollment"]].copy()
doe_small.head()

,Borough,Year,Total Enrollment
0,Bronx,2015-16,"241,986"
1,Bronx,2016-17,"241,776"
2,Bronx,2017-18,"239,955"
3,Bronx,2018-19,"236,267"
4,Bronx,2019-20,"235,448"


In [5]:
# Clean borough names to a consistent format
doe_small["Borough"] = doe_small["Borough"].astype(str).str.strip().str.title()

# Turn "241,986" into 241986.0
doe_small["Total Enrollment"] = (
    doe_small["Total Enrollment"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
)

# Make Year a string
doe_small["Year"] = doe_small["Year"].astype(str)

doe_small.head()


,Borough,Year,Total Enrollment
0,Bronx,2015-16,241986.0
1,Bronx,2016-17,241776.0
2,Bronx,2017-18,239955.0
3,Bronx,2018-19,236267.0
4,Bronx,2019-20,235448.0


In [6]:
# Find latest year in the DOE file, e.g. "2019-20"
latest_year = doe_small["Year"].max()
print("Latest DOE school year:", latest_year)

# Filter to latest year
doe_latest = doe_small[doe_small["Year"] == latest_year]

# Group by borough, sum enrollment
doe_by_borough = (
    doe_latest
      .groupby("Borough", as_index=False)["Total Enrollment"]
      .sum()
)

doe_by_borough


Latest DOE school year: 2019-20


,Borough,Total Enrollment
0,Bronx,235448.0
1,Brooklyn,342332.0
2,Manhattan,180636.0
3,Queens,305623.0
4,Staten Island,67829.0


# Clean New Capacity Program Dataset

In [7]:
sca_small = sca[
    [
        "DISTRICT",
        "BOROUGH",
        "SMALL PS # SEATS",
        "SMALL PS COST",
        "PS/IS # SEATS",
        "PS/IS COST",
        "IS/HS # SEATS",
        "IS/HS COST"
    ]
].copy()

sca_small.head()


,DISTRICT,BOROUGH,SMALL PS # SEATS,SMALL PS COST,PS/IS # SEATS,PS/IS COST,IS/HS # SEATS,IS/HS COST
0,1,MANHATTAN,0,0.00,0,0.00,0,0.0
1,2*,MANHATTAN,"1,368",71.53,"1,782",145.41,0,0.0
2,3,MANHATTAN,0,0.00,692,106.78,0,0.0
3,4,MANHATTAN,0,0.00,0,0.00,0,0.0
4,5,MANHATTAN,245,20.00,0,0.00,0,0.0


In [8]:
sca_small["BOROUGH"] = sca_small["BOROUGH"].astype(str).str.strip().str.title()


In [9]:
# List of all seat and cost columns
seat_cols = ["SMALL PS # SEATS", "PS/IS # SEATS", "IS/HS # SEATS"]
cost_cols = ["SMALL PS COST", "PS/IS COST", "IS/HS COST"]

for col in seat_cols + cost_cols:
    sca_small[col] = (
        sca_small[col]
          .astype(str)
          .str.replace(",", "", regex=False)
          .str.strip()
          .replace({"": pd.NA})
    )
    sca_small[col] = pd.to_numeric(sca_small[col], errors="coerce")

sca_small.head()


,DISTRICT,BOROUGH,SMALL PS # SEATS,SMALL PS COST,PS/IS # SEATS,PS/IS COST,IS/HS # SEATS,IS/HS COST
0,1,Manhattan,0,0.00,0,0.00,0,0.0
1,2*,Manhattan,1368,71.53,1782,145.41,0,0.0
2,3,Manhattan,0,0.00,692,106.78,0,0.0
3,4,Manhattan,0,0.00,0,0.00,0,0.0
4,5,Manhattan,245,20.00,0,0.00,0,0.0


In [10]:
# Total new capacity (all seat types)
sca_small["Total New Capacity"] = (
    sca_small["SMALL PS # SEATS"].fillna(0)
  + sca_small["PS/IS # SEATS"].fillna(0)
  + sca_small["IS/HS # SEATS"].fillna(0)
)

# Total cost (sum of all program types)
sca_small["Total Cost"] = (
    sca_small["SMALL PS COST"].fillna(0)
  + sca_small["PS/IS COST"].fillna(0)
  + sca_small["IS/HS COST"].fillna(0)
)

sca_small.head()


,DISTRICT,BOROUGH,SMALL PS # SEATS,SMALL PS COST,PS/IS # SEATS,PS/IS COST,IS/HS # SEATS,IS/HS COST,Total New Capacity,Total Cost
0,1,Manhattan,0,0.00,0,0.00,0,0.0,0,0.00
1,2*,Manhattan,1368,71.53,1782,145.41,0,0.0,3150,216.94
2,3,Manhattan,0,0.00,692,106.78,0,0.0,692,106.78
3,4,Manhattan,0,0.00,0,0.00,0,0.0,0,0.00
4,5,Manhattan,245,20.00,0,0.00,0,0.0,245,20.00


In [11]:
sca_by_borough = (
    sca_small
      .groupby("BOROUGH", as_index=False)[["Total New Capacity", "Total Cost"]]
      .sum()
)

sca_by_borough


,BOROUGH,Total New Capacity,Total Cost
0,Bronx,5208,499.16
1,Brooklyn,14718,1256.56
2,Manhattan,4087,343.72
3,Queens,18533,1874.47
4,Staten Island,2082,250.64


# Merge Data

In [12]:
# Rename borough columns to a common name before merging
doe_by_borough = doe_by_borough.rename(columns={"Borough": "borough"})
sca_by_borough = sca_by_borough.rename(columns={"BOROUGH": "borough"})

# Merge on borough
merged = doe_by_borough.merge(
    sca_by_borough,
    on="borough",
    how="inner"   # only boroughs that appear in both
)

merged

,borough,Total Enrollment,Total New Capacity,Total Cost
0,Bronx,235448.0,5208,499.16
1,Brooklyn,342332.0,14718,1256.56
2,Manhattan,180636.0,4087,343.72
3,Queens,305623.0,18533,1874.47
4,Staten Island,67829.0,2082,250.64


In [13]:
fig = px.scatter(
    merged,
    x="Total Enrollment",
    y="Total New Capacity",
    color="borough",          # different color for each borough
    size="Total Cost",        # bubble size = total cost of projects
    hover_name="borough",     # show borough name on hover
    labels={
        "Total Enrollment": "Total Student Enrollment",
        "Total New Capacity": "New School Seats",
        "Total Cost": "Total Cost ($)"
    },
    title="Relationship Between Enrollment and New School Capacity by Borough"
)

fig.show()

ValueError: Value of 'hover_data_0' is not the name of a column in 'data_frame'. Expected one of ['borough', 'Total Enrollment', 'Total New Capacity', 'Total Cost'] but received: seats_per_1000_students